In [37]:
#%pip install pandas h5py numpy scipy matplotlib torch torchvision torchaudio -i https://pypi.tuna.tsinghua.edu.cn/simple --break-system-packages

In [38]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

In [39]:
data = sio.loadmat('real2det(1).mat')
Dn = data['real2det']

In [40]:
s1=1
s2=1
def patch2d(A, l1=30, l2=30, s1=s1, s2=s2, mode=1):
    [n1, n2] = A.shape
    if mode == 1:
        tmp = np.mod(n1 - l1, s1)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([s1 - tmp, n2])), axis=0)
        tmp = np.mod(n2 - l2, s2)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([A.shape[0], s2 - tmp])), axis=1)
        [N1, N2] = A.shape
        X = []
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                tmp = np.reshape(A[i1:i1+l1, i2:i2+l2], (l1*l2, 1), order='F')
                X.append(tmp)
        X = np.array(X)
    else:
        pass
    return X[:, :, 0]

def patch2d_inv(X, n1, n2, l1=30, l2=30, s1=s1, s2=s2, mode=1):
    if mode == 1:
        tmp1 = np.mod(n1 - l1, s1)
        tmp2 = np.mod(n2 - l2, s2)
        if tmp1 != 0 and tmp2 != 0:
            A = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
            mask = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
        if tmp1 != 0 and tmp2 == 0:
            A = np.zeros([n1 + s1 - tmp1, n2])
            mask = np.zeros([n1 + s1 - tmp1, n2])
        if tmp1 == 0 and tmp2 != 0:
            A = np.zeros([n1, n2 + s2 - tmp2])
            mask = np.zeros([n1, n2 + s2 - tmp2])
        if tmp1 == 0 and tmp2 == 0:
            A = np.zeros([n1, n2])
            mask = np.zeros([n1, n2])
        [N1, N2] = A.shape
        id = -1
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                id = id + 1
                A[i1:i1+l1, i2:i2+l2] = A[i1:i1+l1, i2:i2+l2] + np.reshape(X[id, :], [l1, l2], order='F')
                mask[i1:i1+l1, i2:i2+l2] = mask[i1:i1+l1, i2:i2+l2] + np.ones([l1, l2])
        A = A / mask
        A = A[0:n1, 0:n2]
    else:
        pass
    return A


In [41]:
#2D data fine-tuning
[nz, nx] = Dn.shape  
w1, w2 = 30, 30
s1, s2 = 1, 1  

X_noisy = patch2d(Dn, w1, w2, s1, s2)
X_noisy = X_noisy.astype(np.float32)

Pdata_noisy = torch.from_numpy(X_noisy)
print(f"Noisy patches shape: {X_noisy.shape}")
print(f"Noisy dtype: {X_noisy.dtype}")

#Data split: 100% training data 
total_patches = len(Pdata_noisy)

finetune_size = int(total_patches * 1.0)   
valid_size = int(total_patches * 0.0)      

finetune_noisy = Pdata_noisy[:finetune_size]
valid_noisy = Pdata_noisy[finetune_size:finetune_size + valid_size]  # empty

print(f"Total number of patches: {total_patches}")
print(f"Fine-tune training set shape: {finetune_noisy.shape} (100%)")
print(f"Validation set shape: {valid_noisy.shape} (0%)")
print(f"Use all data during inference: {Pdata_noisy.shape} (100%)")

# Create TensorDataset
batch_size1 = 64

finetune_data = TensorDataset(finetune_noisy)
valid_data = TensorDataset(valid_noisy)

finetune_loader = DataLoader(finetune_data, batch_size=batch_size1, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=batch_size1, shuffle=False)

print(f"Batch size of fine-tune training dataloader: {len(finetune_loader)}")
print(f"Batch size of validation dataloader: {len(valid_loader)}")

Noisy patches shape: (47817, 900)
Noisy dtype: float32
Total number of patches: 47817
Fine-tune training set shape: torch.Size([47817, 900]) (100%)
Validation set shape: torch.Size([0, 900]) (0%)
Use all data during inference: torch.Size([47817, 900]) (100%)
Batch size of fine-tune training dataloader: 748
Batch size of validation dataloader: 0


In [42]:
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU(0.1, inplace=True)
        self.norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

In [43]:
class SparseAttention(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=2):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        block_size = self.window_size * 2
        num_blocks = max(1, N // block_size)  
        mask = torch.zeros(N, N, device=x.device)
        for i in range(num_blocks):
            start = i * block_size
            end = min(start + block_size, N)  
            mask[start:end, start:end] = 1
        mask = mask.unsqueeze(0).unsqueeze(0).bool()

        attn = attn.masked_fill(~mask, -1e9)
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class Sparse_PatchAE(nn.Module):
    def __init__(self, input_dim, embed_dim=256, depth=1, num_heads=4, window_size=2):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim

        self.embed = nn.Linear(input_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 2, embed_dim) * 0.02)

        self.num_heads = num_heads
        self.window_size = window_size

        self.blocks = nn.ModuleList([
            self._make_block(embed_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Linear(embed_dim, input_dim)
        

    def _make_block(self, dim):
        return nn.ModuleList([
            nn.LayerNorm(dim),
            SparseAttention(dim, self.num_heads, self.window_size),
            nn.LayerNorm(dim),
            nn.Sequential(
                nn.Linear(dim, dim * 4),
                nn.GELU(),
                nn.Linear(dim * 4, dim)
            )
        ])

    def forward(self, x):
        B = x.shape[0]
        x = self.embed(x).unsqueeze(1)  # [B, 1, embed_dim]

        # add cls token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed

        for norm1, attn, norm2, mlp in self.blocks:
            x = x + attn(norm1(x))
            x = x + mlp(norm2(x))

        x = self.norm(x)
        patch_token = x[:, 1, :]
        recon = self.decoder(patch_token)
        return recon


In [44]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()
        self.fcb1 = FCB(input_size, 512, dropout)
        self.Sparse1 = Sparse_PatchAE(512, 512, depth=4)
        
        self.fcb2 = FCB(512, 128, dropout)
        self.Sparse2 = Sparse_PatchAE(128, 128, depth=4)
        
        self.fcb3 = FCB(128, 64, dropout)
        self.Sparse3 = Sparse_PatchAE(64, 64, depth=4)
        
        self.fcb4 = FCB(64, 32, dropout)
        self.Sparse4 = Sparse_PatchAE(32, 32, depth=4)
        
        self.fcb5 = FCB(32, 8, dropout)
        self.Sparse5 = Sparse_PatchAE(8, 8, depth=4)

    def forward(self, x):
        x1 = self.fcb1(x)
        y1 = self.Sparse1(x1)
        x2 = self.fcb2(x1)
        y2 = self.Sparse2(x2)
        x3 = self.fcb3(x2)
        y3 = self.Sparse3(x3)
        x4 = self.fcb4(x3)
        y4 = self.Sparse4(x4)
        x5 = self.fcb5(x4)
        y5 = self.Sparse5(x5)
        return x1, x2, x3, x4, x5, y1, y2, y3, y4, y5

In [45]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.fcb5 = FCB(8 + 8, 32, dropout)
        self.fcb4 = FCB(32 + 32, 64, dropout)
        self.fcb3 = FCB(64 + 64, 128, dropout)
        self.fcb2 = FCB(128 + 128, 512, dropout)
        self.fcb1 = FCB(512 + 512, output_size, dropout)

    def forward(self, x1, x2, x3, x4, x5, y1, y2, y3, y4, y5):
        y51 = torch.cat([x5, y5], dim=1)
        y41 = self.fcb5(y51)
        y41 = torch.cat([y41, y4], dim=1)
        y31 = self.fcb4(y41)
        y31 = torch.cat([y31, y3], dim=1)
        y21 = self.fcb3(y31)
        y21 = torch.cat([y21, y2], dim=1)
        y11 = self.fcb2(y21)
        y11 = torch.cat([y11, y1], dim=1)
        output = self.fcb1(y11)
        return output

In [46]:
class AutoCoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_size, dropout)
        self.decoder = Decoder(output_size, dropout)

    def forward(self, x):
        x1, x2, x3, x4, x5, y1, y2, y3, y4, y5 = self.encoder(x)
        output = self.decoder(x1, x2, x3, x4, x5, y1, y2, y3, y4, y5)
        return output

In [47]:
def huber_loss(pred, target, delta=1.35):
    error = pred - target
    abs_error = torch.abs(error)
    quadratic = torch.clamp(abs_error, max=delta)
    linear = abs_error - quadratic
    loss = 0.5 * quadratic ** 2 + delta * linear
    return loss.mean()

In [48]:
#Model initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Loading universal denoiser weights (pre-trained 2D model)
model = AutoCoder(input_size=w1*w2, output_size=w1*w2, dropout=0.1).to(device)
model.decoder.fcb1 = torch.nn.Linear(1024, w1*w2).to(device)

# Loading previously trained universal weights
model.load_state_dict(torch.load('best_model_30,30_final.pth', map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu')))
print("Universal denoiser weights loaded successfully")

# Fine-tuning parameters configuration
optimizer = torch.optim.Adam(model.parameters(), lr=0.0004) 
num_epochs = 3
# Random masking ratio
mask_ratio = 0.25  

#Training loop
print("Start Fine-tuning on Target Area...")
prev_train_loss = float('inf')
loss_train = []
loss_valid = []

for epoch in range(num_epochs):
    
    model.train()
    train_loss = 0.0
    for batch in finetune_loader:
        noisy_patch = batch[0].to(device)
        
        # Generate a [0, 1) uniform random number matrix with the same shape as the input
        rand_tensor = torch.rand_like(noisy_patch)
        random_mask_2d = rand_tensor < mask_ratio
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch
        
        optimizer.zero_grad()
        outputs = model(input_patch)
        
        #Loss calculation: only focus on masked scatter points
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.35)
        loss = rec_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * input_patch.size(0)
    
    train_loss = train_loss / len(finetune_noisy)
    loss_train.append(train_loss)
    
    #Validation phase (automatically skipped because valid_loader is empty)
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for batch in valid_loader:  
            valid_patch = batch[0].to(device)
            
            rand_tensor_val = torch.rand_like(valid_patch)
            random_mask_2d_val = rand_tensor_val < mask_ratio
            input_val_patch = valid_patch * (~random_mask_2d_val)
            
            outputs = model(input_val_patch)
            rec_loss_val = huber_loss(outputs[random_mask_2d_val], valid_patch[random_mask_2d_val], delta=1.35)
            
            valid_loss += rec_loss_val.item() * valid_patch.size(0)
    
    if len(valid_noisy) > 0:
        valid_loss = valid_loss / len(valid_noisy)
    else:
        valid_loss = None
    
    loss_valid.append(valid_loss)
    

    if valid_loss is not None:
        print(f"Epoch[{epoch+1}/{num_epochs}], Train Loss: {train_loss:.6f}, Valid Loss: {valid_loss:.6f}")
    else:
        print(f"Epoch[{epoch+1}/{num_epochs}], Train Loss: {train_loss:.6f} ")
    
    
    # Saving best model (based on training loss)
    if train_loss < prev_train_loss:
        prev_train_loss = train_loss
        torch.save(model.state_dict(), 'syn2d_best_Fine-tune training.pth')
        print(f" Updating best model，Train Loss: {train_loss:.6f}")

print("=" * 60)
print("Fine-tuning completed！")
print(f"Total training {num_epochs} epoch")
print(f"Saved weight file:")


Universal denoiser weights loaded successfully
Start Fine-tuning on Target Area...
Epoch[1/3], Train Loss: 0.306575 
 Updating best model，Train Loss: 0.306575
Epoch[2/3], Train Loss: 0.250568 
 Updating best model，Train Loss: 0.250568
Epoch[3/3], Train Loss: 0.233554 
 Updating best model，Train Loss: 0.233554
Fine-tuning completed！
Total training 3 epoch
Saved weight file:


In [49]:
#### Inference and post-processing
import scipy 
# Loading the best fine-tuned model
model = AutoCoder(input_size=w1*w2, output_size=w1*w2, dropout=0.1).to(device)
model.decoder.fcb1 = torch.nn.Linear(1024, w1*w2).to(device)
model.load_state_dict(torch.load('syn2d_best_Fine-tune training.pth', map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu')))
model.eval()

# Full-data inference
with torch.no_grad():
    Pdata_noisy = Pdata_noisy.to(device)
    outputs = model(Pdata_noisy)
    
    # Convert to numpy array for fine processing
    outputs = outputs.cpu().numpy()

    
print("Inference output statistics")
print(f"Total number of output patches: {outputs.shape[0]}")
print(f"Number of non-zero patches: {np.count_nonzero(outputs.any(axis=1))}")
print(f"Output maximum value: {outputs.max():.4f}, minimum value: {outputs.min():.4f}, mean value: {outputs.mean():.4f}")

# Inverse blocking to merge into a complete image
denoised_data = patch2d_inv(outputs, n1=nz, n2=nx, l1=w1, l2=w2, s1=s1, s2=s2)

# Save data
scipy.io.savemat("syn2d_best_Fine-tune training.mat", {"D_denoised": denoised_data})
print("✅ File saved：syn2d_best_Fine-tune training.mat")
print(f"✅ Denoised data shape: {denoised_data.shape}")

Inference output statistics
Total number of output patches: 47817
Number of non-zero patches: 47817
Output maximum value: 7.4164, minimum value: -8.6508, mean value: -0.0088
✅ File saved：syn2d_best_Fine-tune training.mat
✅ Denoised data shape: (512, 128)
